# Day 1 — Multi-Step Retrieval & Planner/Executor Flows

**Module 6 · Agentic RAG Testing**

---

## What we'll cover today

| # | Topic | Why it matters |
|---|---|---|
| 1 | From one retrieval to many | Agentic RAG adds a planner that can loop — that loop is the whole new surface area |
| 2 | Real incident: Klarna's AI rollback | Simple queries matched humans; complex multi-step disputes didn't |
| 3 | New failure modes | Infinite loops, premature stops, query drift — none of these exist in a single-pass system |
| 4 | How this changes what you test | Module 5's metrics still apply, plus a new axis pointed at the loop itself |
| 5 | Equivalence partitioning by hop count | Same Module 4 Day 4 technique, new dimension |
| 6 | Building and tracing a 2-hop loop | See the planner's decision at every hop, not just the final answer |

**Estimated time:** 60 minutes
**Run order:** top to bottom. The loop cell calls a real deployed LLM — copy `../.env.example` to `.env` (this `examples/` folder) and fill in your credentials first. `@traceable` itself still works with no LangSmith key configured; tracing is just skipped.

---

> **Where we are in the course**
> Module 4 Day 4 gave you equivalence partitioning, boundary value analysis, the coverage matrix, and hard negatives.
> Module 5 gave you the retriever/generator split, RAGAS's 4 core metrics, and LangSmith tracing for a single-pass pipeline.
> Today the pipeline gets a planner that can decide to retrieve more than once before answering — same testing toolkit, one new component to point it at.

---
## From one retrieval to many

Module 5's flow was a straight line:

```
User question -> RETRIEVER -> GENERATOR -> answer
```

Agentic RAG adds a decision point that can loop:

```
User question
    │
    ▼
PLANNER  →  do I have enough information to answer? if not, what should I search for next?
    │
    ├─ NOT ENOUGH  →  reformulate query  →  RETRIEVER  →  back to PLANNER
    │
    └─ ENOUGH      →  GENERATOR  →  final answer
```

> **Plain English:** Module 5's system was a librarian who fetches one book and writes a summary. An agentic RAG system is a research assistant who reads the first book, realizes it raises a follow-up question, goes back for a second book, and only writes the summary once they actually have what they need. The second assistant is more capable — and has far more ways to go wrong before they ever start writing.

---
## Real incident: Klarna's AI customer service rollback (May 2025)

Klarna replaced roughly 700 customer-service roles with an OpenAI-powered AI assistant, reporting in early 2024 that it handled two-thirds of chats with under-2-minute resolution times. By May 2025, CEO Sebastian Siemiatkowski publicly walked the rollout back and resumed hiring human agents. The detail that matters here: **AI matched human performance on simple queries** (order status, payment schedules) **but quality dropped noticeably on complex cases** — disputes, fraud claims, hardship cases — exactly the questions that require chaining several pieces of information together rather than answering from one lookup.

> **Why this matters for today:** "simple query" is a Module 5 problem — one retrieval, one answer. "Complex dispute" is a Module 6 problem — it needs the planner to recognize that one retrieved fact isn't enough, go fetch more, and reason across all of it. Klarna's gap between the two is the exact gap this module tests for *before* a rollout, not after.

---
## New failure modes that only exist in multi-step retrieval

| Failure mode | What it looks like |
|---|---|
| **Infinite retrieval loop** | The planner never decides it has enough information; it keeps reformulating and re-querying |
| **Premature stop** | The planner answers after 1 hop when the question genuinely needed 2-3 |
| **Query drift** | Each reformulated query moves further from the user's actual intent |
| **Reasoning chain break** | Every needed fact was retrieved correctly across hops, but combined incorrectly at the end (Day 2 builds this one) |

None of these can happen in Module 5's single-pass flow — they only exist because the system can now make a *decision* about whether to continue.

---
## How this changes what you actually test

Everything Module 5 taught you about testing a RAG system's *output* — faithfulness, answer relevancy, context precision, context recall — still applies here, unchanged, and you'll use it again on Day 2. What's new is that there's now a *process* in between the question and the answer, and a process has its own way of being wrong that has nothing to do with whether the final text is grounded:

| Testing dimension | Module 5 (single-pass RAG) | Module 6 (agentic RAG) |
|---|---|---|
| Faithfulness / answer relevancy | one retrieval, one response to score | still applies — scored against the FINAL response and the FULL accumulated fact set |
| Context precision / recall | one retrieval call to score | same metrics, but now computed on a *union* across hops — they can't tell you which hop was the bad one |
| How many times retrieval ran | N/A — always exactly 1 | **New.** `num_hops`, `hit_max_hops` — nothing in Module 5 needed this, because there was nothing to count |
| Whether the planner's reasoning was sound | N/A — no planner exists | **New, and no RAGAS metric covers it.** You read `reasoning` directly, or check whether the loop terminated with confidence |
| Combining facts from multiple hops correctly | N/A — only ever one fact set, nothing to combine | **New — Day 2's `reasoning_chain_break`.** Faithfulness can't see this failure mode, because it can't exist when there's only ever been one retrieval |
| Tracing | Nice-to-have — one retrieval, one generation, not much to get lost | **Load-bearing.** Without per-hop spans you cannot tell a premature stop from query drift from a clean success — they can look identical from the final answer alone |

The short version: agentic RAG testing is Module 5's testing **plus** a new axis pointed at the loop itself, not Module 5's testing done differently. Reusing Module 5's RAGAS suite unchanged here would correctly catch generation-level problems and completely miss the loop-level ones — that gap is what the rest of this module builds checks for, starting with `num_hops` and `hit_max_hops` below.


---
## Equivalence partitioning: hops required

Same technique from Module 4 Day 4, new dimension.

In [4]:
hop_partitions = {
    "hops_required": {
        "single_hop":      "the answer is fully contained in one retrievable chunk",
        "two_hop":         "the answer requires combining facts from two separate retrievals",
        "three_plus_hop":  "the answer requires chaining three or more retrievals",
    },
}

for name, description in hop_partitions["hops_required"].items():
    print(f"[{name:<16}] {description}")

print()
print("A test suite built only from 'single_hop' cases will pass beautifully and tell you")
print("nothing about whether your planner can handle a Klarna-style dispute — the same")
print("'happy path only' trap from Module 4 Day 4, one layer up.")

[single_hop      ] the answer is fully contained in one retrievable chunk
[two_hop         ] the answer requires combining facts from two separate retrievals
[three_plus_hop  ] the answer requires chaining three or more retrievals

A test suite built only from 'single_hop' cases will pass beautifully and tell you
nothing about whether your planner can handle a Klarna-style dispute — the same
'happy path only' trap from Module 4 Day 4, one layer up.


---
## Building and tracing a real 2-hop loop

A small in-memory corpus where the answer genuinely requires two hops: *"What is the cancellation fee for the product that replaced WidgetPro 2000?"* You first need to find out WHAT replaced it, then look up THAT product's fee.

Retrieval, the planner's continue/stop decision, and the final answer are all real calls now — `agent.py` (right next to this notebook) embeds the corpus and each query, asks a live LLM whether it has enough information (and what to search for next if not), and asks a live LLM to write the answer. Nothing here is scripted. Needs `.env` configured with your deployed LLM's credentials — the same resource Module 5 used.


In [6]:
from agent import CORPUS, agentic_rag

print("Corpus:")
for doc in CORPUS:
    print(f"  - {doc}")
print()

result = await agentic_rag(
    "What is the cancellation fee for the product that replaced WidgetPro 2000?",
    verbose=True,
)
print(f"[{result.num_hops} hop(s), facts used: {result.retrieved_contexts}")
print()
print("Answer:", result.response)


Corpus:
  - WidgetPro 2000 was discontinued in 2023 and replaced by WidgetPro 3000.
  - WidgetPro 3000's cancellation fee is $0 -- it can be canceled anytime at no charge.
  - WidgetPro 2000's cancellation fee was $50 before it was discontinued.
  - Our premium support plan includes 24/7 phone access and a 1-hour response SLA.
  - TurboMax 5 was renamed to TurboMax Pro in 2024 after a rebranding update.
  - TurboMax Pro's annual subscription costs $299, unchanged from TurboMax 5's price.
  - Our headquarters relocated from Austin to Denver in 2022.
  - The Denver office does not offer walk-in support; all support is online only.

[hop 1] query='What is the cancellation fee for the product that replaced WidgetPro 2000?'
[hop 1] retrieved: ["WidgetPro 2000's cancellation fee was $50 before it was discontinued.", "WidgetPro 3000's cancellation fee is $0 -- it can be canceled anytime at no charge."]
[hop 1] planner.enough_info=False  reasoning="The question asks for the cancellation fee fo

With `LANGSMITH_TRACING` enabled (set it in `.env` — see `../.env.example`), this shows up at [smith.langchain.com](https://smith.langchain.com) as a single `agentic_rag_loop` trace with nested `retriever`, `planner`, and `generator` spans per hop — the same information the `verbose=True` output above prints inline, just persisted and inspectable after the fact. Compare a hop's printed `reasoning` to what a rule-based stand-in would have had to hard-code — that's the difference a real planner makes.

Notice `hit_max_hops` in the printed result above: it's `False` here because the planner itself confirmed `enough_info=True` before hops ran out. `AgentResult` tracks this honestly — it's the bounded-loop shape of Day 1's "infinite retrieval loop" failure mode, and when it's `True`, `agent.py`'s generator is told explicitly that the planner wasn't confident, so it should hedge rather than confidently answer from an incomplete fact set (see `_generate`'s `uncertain` parameter).


---
## Try It Yourself

1. **Force a premature stop and watch `hit_max_hops` flip:** call `await agentic_rag(question, max_hops=1, verbose=True)` on the WidgetPro question above. With only one hop allowed, the planner never gets a second chance to say "not enough" — confirm `result.hit_max_hops` is now `True`. What does the generator do with an incomplete fact set now that it's been told the planner was uncertain? Does it hedge, or guess anyway?
2. **Read the planner's real reasoning:** run the WidgetPro question again with `verbose=True` and read `reasoning` at each hop. Does it match what you'd expect a human to say? Now try a question the corpus can't answer at all (e.g. `"What is the CEO's phone number?"`) and watch how many hops it takes before the planner gives up — confirm `hit_max_hops` is `True` in that case too, and read how the generator's answer differs from a normal run.
3. **Add a third hop:** add two new corpus entries to `CORPUS` in `agent.py` that require chaining three facts together (e.g. a product renamed, then its price changed, then a new discount applied), and ask a 3-hop question. Does the real planner correctly take 3 hops (`hit_max_hops=False`), or does it stop early — or run past `max_hops` (`hit_max_hops=True`) without ever setting `enough_info=True`?

Exercise file: [`exercises/01_multistep_retrieval_exercise.md`](../exercises/01_multistep_retrieval_exercise.md)


---
## Summary

### What we built today
- The planner/executor flow that distinguishes agentic RAG from Module 5's single-pass pipeline
- A real incident (Klarna) mapped directly onto the single-hop/multi-hop quality gap
- 4 new failure modes that only exist once retrieval can loop
- An equivalence partition by `hops_required`, reusing Module 4 Day 4's technique
- A traced, working 2-hop loop — real embedding retrieval, real LLM planner, real LLM generator — you can deliberately push into 3 different failure shapes

### Carried forward unchanged
Equivalence partitioning still works the same way it did in Module 4 Day 4 and Module 5 Day 2 — point it at a new dimension (hop count) and it surfaces the same kind of accidental happy-path-only coverage gap.

**Next:** Day 2 — Tool/Memory Validation, Multi-Hop Reasoning & Failure-Path Testing, where we build hard negatives for the failure modes that hide *inside* a clean-looking multi-hop trace.

---